<a href="https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/01_first_deep_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/01_first_deep_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 · Your first Deep Agent

You are going to build a research assistant that plans its own work, searches the web, takes
notes into a filesystem, and writes a report — in about ten lines of code.

**New in this lesson:** `create_deep_agent`, the built-in tool suite, tracing, LangSmith Studio.

> **Need a key?** You need a LangSmith API key stored in Colab Secrets (🔑 in the left
> sidebar) as `LANGSMITH_API_KEY`, with **"Notebook access" turned on**. If you have not done
> that yet, run **[00 · Setup](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/00_setup.ipynb)** first — it takes 10 minutes and
> checks everything.

In [ ]:
# --- snippet:setup v1 ---
%pip install -qq \
  "deepagents~=0.7.6" \
  "langchain~=1.3.15" \
  "langchain-openai~=1.5.1" \
  "langsmith~=0.11.0" \
  "git+https://github.com/langchain-samples/langsmith-studio-nb.git"

import os

try:
    from google.colab import userdata

    key = userdata.get("LANGSMITH_API_KEY")
except Exception:  # not on Colab, or secret unavailable
    from getpass import getpass

    key = os.environ.get("LANGSMITH_API_KEY") or getpass("LANGSMITH_API_KEY: ")

os.environ["LANGSMITH_API_KEY"] = key
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "lcw-01-first-agent"

# One constant, used everywhere. Models are served by the LangSmith gateway,
# so this key is the only credential the notebook needs.
MODEL = "langsmith:openai/gpt-5.6-luna"
# --- /snippet ---

print("Ready.")

## 2. Build the agent

`create_deep_agent` takes a model, some tools, and instructions. Everything else — planning,
files, delegation — comes with it.

The agent comes pre-built with a tool calling loop and some useful middleware.

In [ ]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    model=MODEL,
    system_prompt=(
        "You are a research assistant.\n"
        "Research the user's question thoroughly using web search.\n"
        "Keep working notes in files as you go.\n"
        "Always write your final report to report.md before you finish.\n"
        "Cite sources inline."
    ),
    tools=[{"type": "web_search"}],  # a built-in tool part of OpenAI
)
agent

## 3. Run it in LangSmith Studio

LangSmith Studio is a visual debugger for agents. It runs against the agent you just built.

In [ ]:
from langsmith_studio_nb import start_studio

# Serves the notebook variable named `agent` and prints a link.
start_studio("agent")

## 4. What is a harness?

```
Agent = Model + Harness
```

The harness is the scaffolding around the model that connects it to the real world.

1. An agent is only as good as the context provided to the model
2. The job of a harness is to provide context to the model at every step

So, to build a useful agent, you need a harness that’s great at delivering the right context for the given task to the model.



`create_deep_agent` gives you a production-ready foundation. Here's an example of some of the key customizations we will look at throughout this workshop.

In [ ]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    model="langsmith:openai/gpt-5.6-luna",  # Any provider, any model (reasoning models work best)
    system_prompt="You are a helpful assistant.",
    tools=[{"type": "web_search"}], # Additional tools
    memory=["./AGENTS.md"],
    skills=["./skills/"],
    middleware=[...],
    backend=...,
    subagents=[...],
)

---

## 📌 Key takeaways

- When building an agent, start with `create_deep_agent`. It is the fastest way to get started with a lot of best practices built-in.
- Then focus on customizing the **harness** with the data, behaviors, and capabilities your use case needs.
- Use LangSmith Studio to test and debug your agent.

---

## ➡️ Next

**[02 · Files as the agent's workspace](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/02_backends.ipynb)**

The agent wrote files. Where did they actually go — and what happens to them when the
conversation ends? That question turns out to be the most important design decision in an agent.